In [2]:
import gradio as gr
import requests
import os 
import dotenv

dotenv.load_dotenv()
AZURE_FOUNDRY_KEY = os.getenv('AZURE_FOUNDRY_KEY')

## 화면 구성

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown('🖼️이미지 분석')

    image_url_textbox = gr.Textbox(label='Image URL')
    send_button = gr.Button("전송")
    output_image = gr.Image(label="출력이미지", interactive=False)

demo.launch()

# 로컬이 아닌 image URL에서 가져오는 것이므로 
# 해당 url에서 이미지를 요청해서 가져오는 과정이 하나 더 필요함 


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 이미지 url에서 가져오기

In [5]:
response= requests.get(
    "https://learn.microsoft.com/ko-kr/azure/ai-services/computer-vision/images/windows-kitchen.jpg"
)

# response= requests.get(
#     "https://learn.microsoft.com/ko-kr/azure/ai-services/computer-vision/images/windows-kitchen.jpg",
#     stream=True 
# )
    # stream = True면 raw형태로 받아올 수 있음 but 권장X


print(response.content)  
# _content가 아니라 content를 반환해도 되는 이유
# requests 라이브러리에서 response.content라는 게 정의되어 있기 때문임
# with open("img", "wb") as f:
#         f.write(response.content)

b'\xff\xd8\xff\xe1\x00\x18Exif\x00\x00II*\x00\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xec\x00\x11Ducky\x00\x01\x00\x04\x00\x00\x00d\x00\x00\xff\xe1\x03,http://ns.adobe.com/xap/1.0/\x00<?xpacket begin="\xef\xbb\xbf" id="W5M0MpCehiHzreSzNTczkc9d"?> <x:xmpmeta xmlns:x="adobe:ns:meta/" x:xmptk="Adobe XMP Core 5.6-c140 79.160451, 2017/05/06-01:08:21        "> <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"> <rdf:Description rdf:about="" xmlns:xmp="http://ns.adobe.com/xap/1.0/" xmlns:xmpMM="http://ns.adobe.com/xap/1.0/mm/" xmlns:stRef="http://ns.adobe.com/xap/1.0/sType/ResourceRef#" xmp:CreatorTool="Adobe Photoshop CC (Macintosh)" xmpMM:InstanceID="xmp.iid:8FDF1C2FB47E11E89A7BE3BD75E9A10E" xmpMM:DocumentID="xmp.did:8FDF1C30B47E11E89A7BE3BD75E9A10E"> <xmpMM:DerivedFrom stRef:instanceID="xmp.iid:8FDF1C2DB47E11E89A7BE3BD75E9A10E" stRef:documentID="xmp.did:8FDF1C2EB47E11E89A7BE3BD75E9A10E"/> </rdf:Description> </rdf:RDF> </x:xmpmeta> <?xpacket end="r"?>\xff\xee\x00\x

#### `response._content` vs `response.content`

`requests` 라이브러리에서 `response.content`는 **property**로 정의되어 있고, 내부적으로 `_content`를 반환하기 때문에 둘 다 작동합니다.


| 구분 | `response._content` | `response.content` |
|---|---|---|
| 타입 | private 변수 (직접 접근) | public property |
| 내부 동작 | 값을 그냥 반환 | lazy loading 후 `_content` 반환 |
| 권장 여부 | X (내부 구현 세부사항) | O (공식 API) |

### `_content`를 직접 쓰면 안 되는 이유

- `_` prefix는 Python 관례상 "내부용이니 건드리지 마세요"라는 의미
- 응답이 아직 읽히지 않은 경우 `_content`는 `False`일 수 있어서 예상치 못한 결과가 나올 수 있음
- 라이브러리 버전이 바뀌면 내부 구현이 달라질 수 있음

항상 `response.content`를 사용하는 것이 맞습니다.


## Event listener 구상
- click_send 구현
- 받아온 바이너리 파일 PIL 형태로 변환

In [1]:
import requests
import gradio as gr
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont

with gr.Blocks() as demo:
    gr.Markdown('🖼️이미지 분석')

    # 이미지 url 전송 이벤트 
    def click_send(image_url):
        response = requests.get(image_url)
        image = Image.open(BytesIO(response.content))
         # 오픈이 b가 존재하는 게 아닌 완전한 binary형태 byteIO로 변환해서 open이 받을 수 있게 함
        return image
    
    image_url_textbox = gr.Textbox(label='Image URL')
    send_button = gr.Button("전송")
    output_image = gr.Image(label="출력이미지", interactive=False)

    # 이미지 전송 
    send_button.click(click_send, inputs=[image_url_textbox], outputs=[output_image])

demo.launch()

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Object Detection

- 모델 API 만들기
- bounding box 그리기

In [ ]:
import requests
import gradio as gr
from io import  BytesIO
from PIL import Image, ImageDraw, ImageFont
import os 
import dotenv
dotenv.load_dotenv()
AZURE_FOUNDRY_KEY = os.getenv('AZURE_FOUNDRY_KEY')
import random

# 바운딩박스 색상 랜덤 지정
def random_color():
    return(random.randint(0,255), random.randint(0,255), random.randint(0,255))

def request_image_analysis(image_url, features=['objects']):
    # endpoint = "https://9ai043-foundry.services.ai.azure.com/computervision/imageanalysis:analyze?api-version=2024-02-01&features=objects&language=en"
 
    # 나중에 feature를 추가하거나 변경하기 쉽게 params로 분리
    endpoint = "https://9ai043-foundry.services.ai.azure.com/computervision/imageanalysis:analyze?"

    params = {
        "api-version": "2024-02-01",
        "features": ",".join(features),
        "language": "en"
    }

    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_FOUNDRY_KEY
    }

    body = {
        "url": image_url
    }

    response = requests.post(endpoint, params= params, headers=headers, json=body)

    if not response.ok:
        print(f"ERROR: {response.status_code}")
        return None
    
    
    response_json = response.json()
    print(response_json)
    
    
    return response_json #일단 요청 API 응답 전부 들고 오기, draw단에서 처리할 것임

def draw_image(image_url, result_data):
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content))
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=20)

    # 분석 요청 했던 결과를 여기서 가공
    # 예외처리 object가 선택된 경우에만 바운딩박스그리기
    if 'objectsResult' in result_data:
        item_list = result_data['objectsResult']['values']
        for item in item_list:
            color = random_color()
            bounding_box = item['boundingBox']
            x, y, w, h = (
                bounding_box["x"],
                bounding_box["y"],
                bounding_box["w"],
                bounding_box["h"]
            )
            tag = item['tags'][0]
            name = tag.get('name')
            confidence = tag.get('confidence')

            # confidence 임계값 설정 예외처리 
            if confidence < 0.7:
                continue

            draw.rectangle([(x,y), (x+w, y+h)], outline=color, width=2)
            draw.text(
                (x+10, y),
                "{}({:.2f}%)".format(name, confidence * 100),
                fill =color,
                font=font
            )
            print(name, "{:.2f}%".format(confidence * 100), x, y, w, h )

    return image

### feature가 어떻게 들어오는지 테스트
# def change_features(selected_features):
#     print(selected_features)

with gr.Blocks() as demo:
    gr.Markdown('🖼️이미지 분석')

    # 이미지 url 전송 이벤트 
    def click_send(image_url, selected_features):
        response_data = request_image_analysis(image_url, selected_features)
        #단순 이미지 열기(테스트)
        # image = Image.open(BytesIO(response.content)) 
        # 이미지에 box그린 이미지 
        image = draw_image(image_url, response_data)
         # 오픈이 b가 존재하는 게 아닌 완전한 binary형태 byteIO로 변환해서 open이 받을 수 있게 함
        return image, response_data
    
    # feature 선택 옵션 (분석 옵션 추가)
    features_list = ["objects", "tags","caption","read", "denseCaptions",'smartCrops',"people"]
    features_checkbox = gr.CheckboxGroup(label="기능 선택", choices=features_list)

    image_url_textbox = gr.Textbox(label='Image URL')
    send_button = gr.Button("전송")

    with gr.Row():
        output_image = gr.Image(label="출력이미지", interactive=False)
        output_json = gr.JSON(label="출력 데이터")


    # 이미지 전송 
    send_button.click(click_send, inputs=[image_url_textbox, features_checkbox], outputs=[output_image, output_json])
    # feature 변경
    # features_checkbox.change(change_features, inputs=[features_checkbox])

demo.launch()

# 이미지 분석 API 테스트
# request_image_analysis("https://learn.microsoft.com/azure/cognitive-services/computer-vision/images/windows-kitchen.jpg")

# 이미지 전송 테스트
# click_send("https://learn.microsoft.com/azure/cognitive-services/computer-vision/images/windows-kitchen.jpg")

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


{'modelVersion': '2023-10-01', 'captionResult': {'text': 'a person using a laptop', 'confidence': 0.8262178897857666}, 'metadata': {'width': 1260, 'height': 473}, 'tagsResult': {'values': [{'name': 'computer', 'confidence': 0.9865934252738953}, {'name': 'clothing', 'confidence': 0.9695653915405273}, {'name': 'laptop', 'confidence': 0.9658201932907104}, {'name': 'person', 'confidence': 0.9536289572715759}, {'name': 'indoor', 'confidence': 0.9420197010040283}, {'name': 'wall', 'confidence': 0.8871886730194092}, {'name': 'woman', 'confidence': 0.8632704019546509}, {'name': 'using', 'confidence': 0.5603535771369934}]}}


## 옵션 선택하기
- 옵션 별 색상 다르게 지정
- 해당 옵션이 선택됐을 때만 해당 박스/데이터를 표시
- T성별 중립 옵션 추가, 컴포넌트 추가

In [28]:
import requests
import gradio as gr
from io import  BytesIO
from PIL import Image, ImageDraw, ImageFont
import os 
import dotenv
dotenv.load_dotenv()
AZURE_FOUNDRY_KEY = os.getenv('AZURE_FOUNDRY_KEY')
import random

# 바운딩박스 색상 랜덤 지정
def random_color():
    return(random.randint(0,255), random.randint(0,255), random.randint(0,255))

def request_image_analysis(image_url, features=['objects'], options=dict()):
    # endpoint = "https://9ai043-foundry.services.ai.azure.com/computervision/imageanalysis:analyze?api-version=2024-02-01&features=objects&language=en"
 
    # 나중에 feature를 추가하거나 변경하기 쉽게 params로 분리
    endpoint = "https://9ai043-foundry.services.ai.azure.com/computervision/imageanalysis:analyze?"

    params = {
        "api-version": "2024-02-01",
        "features": ",".join(features),
        "language": "en"
    }

    params.update(options) # 옵션이 존재할 때 params에 추가

    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_FOUNDRY_KEY
    }

    body = {
        "url": image_url
    }

    response = requests.post(endpoint, params= params, headers=headers, json=body)

    if not response.ok:
        print(f"ERROR: {response.status_code}")
        return None
    
    
    response_json = response.json()
    print(response_json)
    
    
    return response_json #일단 요청 API 응답 전부 들고 오기, draw단에서 처리할 것임

def draw_image(image_url, result_data):
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content))
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=20)

    # 분석 요청 했던 결과를 여기서 가공
    DRAWABLE_FEATURES = ['objectsResult', 'denseCaptionsResult', 'smartCropsResult']
    for feature in DRAWABLE_FEATURES:
        result = result_data.get(feature, None)
        if result is None:
            continue
        print(result)
        color = random_color()  
    # 예외처리 object, denseCaptions, SmartCropsResult가 선택된 경우에만 바운딩박스그리기
    # if 'objectsResult' in result_data:
        item_list = result_data[feature]['values']
        for item in item_list:
            # color = random_color()
            bounding_box = item['boundingBox']
            x, y, w, h = (
                bounding_box["x"],
                bounding_box["y"],
                bounding_box["w"],
                bounding_box["h"]
            )
            # tag = item['tags'][0]
            # name = tag.get('name')
            # confidence = tag.get('confidence')

            # confidence 임계값 설정 예외처리 
            # if confidence < 0.7:
            #     continue

            draw.rectangle([(x,y), (x+w, y+h)], outline=color, width=2)
            text =""
            # draw.text(
            #     (x+10, y),
            #     "{}({:.2f}%)".format(name, confidence * 100),
            #     fill =color,
            #     font=font
            # )
            # print(name, "{:.2f}%".format(confidence * 100), x, y, w, h )

            if feature == 'objectsResult':
                tag = item['tags'][0]
                name = tag.get('name')
                confidence = tag.get('confidence')
                text = "{}({:.2f}%)".format(name, confidence * 100)
            elif feature == 'denseCaptionsResult':
                text = item["text"]
            elif feature == 'smartCropsResult':
                text="{:.2f}".format(item['aspectRatio'])
            
            draw.text(
                (x+10, y),
                text,
                fill =color,
                font=font
            )
           
            # draw.text((x+10, y), feature, fill=color, font=font)

    return image

### feature가 어떻게 들어오는지 테스트
def change_features(selected_features):
    print(selected_features)
    selected_caption = False
    selected_smartcrop = False

    if "caption" in selected_features or "denseCaptions" in selected_features:
        selected_caption = True
    
    if "smartCrops" in selected_features:
        selected_smartcrop = True
    
    return gr.update(visible=selected_caption, interactive=True), gr.update(visible=selected_smartcrop, interactive=True)


with gr.Blocks() as demo:
    gr.Markdown('🖼️이미지 분석')

    # 이미지 url 전송 이벤트 
    def click_send(image_url, selected_features, is_neutral, smartcrops_text):
        # response_data = request_image_analysis(image_url, selected_features)
        # #단순 이미지 열기(테스트)
        # # image = Image.open(BytesIO(response.content)) 
        # # 이미지에 box그린 이미지 
        # image = draw_image(image_url, response_data)
        #  # 오픈이 b가 존재하는 게 아닌 완전한 binary형태 byteIO로 변환해서 open이 받을 수 있게 함
        # return image, response_data

        # return image

        # 옵션 확인
        print(is_neutral, smartcrops_text)
        option = dict()

        if "caption" in selected_features or "denseCaptions" in selected_features:
            option.update({"gender-neutral-caption": is_neutral})
    
        if "smartCrops" in selected_features:
            option.update({"smartcrops-aspect-ratio": smartcrops_text})
            selected_smartcrop = True

        response_data = request_image_analysis(image_url, selected_features, options=option )
        image = draw_image(image_url, response_data)
        return image, response_data
    
    # feature 선택 옵션 (분석 옵션 추가)
    features_list = ["objects", "tags","caption","read", "denseCaptions",'smartCrops',"people"]
    features_checkbox = gr.CheckboxGroup(
        label="기능 선택",
        choices=features_list,
        value=['objects'])

    # 성별 중립 옵션 컴포넌트 추가
    gender_radio = gr.Radio(label="성별 중립 옵션", choices=[('중립', True), ('구분', False)], value=False, visible=False, interactive=True)  # 기본값은 Off로 설정

    with gr.Column(visible=False) as smartcrop_container:
        # smartcrop 크기 지정
        smartcrop_textbox = gr.Textbox(label="smarCrops 크기 지정", placeholder="ex)0.75, 1.2, 1.7")
    
    
    
    image_url_textbox = gr.Textbox(label='Image URL')
    send_button = gr.Button("전송")
    with gr.Row():
        output_image = gr.Image(label="출력이미지", interactive=False)
        output_json = gr.JSON(label="출력 데이터")


    # 이미지 전송 
    send_button.click(click_send, inputs=[image_url_textbox, features_checkbox], outputs=[output_image, output_json])
    # feature 변경
    features_checkbox.change(change_features, inputs=[features_checkbox], outputs=[gender_radio, smartcrop_container])
    
demo.launch()

# 이미지 분석 API 테스트
# request_image_analysis("https://learn.microsoft.com/azure/cognitive-services/computer-vision/images/windows-kitchen.jpg")

# 이미지 전송 테스트
# click_send("https://learn.microsoft.com/azure/cognitive-services/computer-vision/images/windows-kitchen.jpg",
#            ["objects", "tags","caption","read", "denseCaptions",'smartCrops',"people"])

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\utils.py:1177: UserWarning: Expected 4 arguments for function <function click_send at 0x000002B8FF4AACA0>, received 2.
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\utils.py:1181: UserWarning: Expected at least 4 arguments for function <function click_send at 0x000002B8FF4AACA0>, received 2.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


['objects', 'smartCrops']


Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 2169, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 1928, in postprocess_data
    state[block._id] = block.__class__(**kwargs)
                       ^^^^^^^^^^^^^^^^^^^

['objects']


...

Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 2169, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 1928, in postprocess_data
    state[block._id] = block.__class__(**kwargs)
                       ^^^^^^^^^^^^^^^^^^^

['smartCrops']


Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 2169, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 1928, in postprocess_data
    state[block._id] = block.__class__(**kwargs)
                       ^^^^^^^^^^^^^^^^^^^

[]


Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 2169, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 1928, in postprocess_data
    state[block._id] = block.__class__(**kwargs)
                       ^^^^^^^^^^^^^^^^^^^

In [ ]:
## 정답 파일

# 필요한 라이브러리들을 임포트합니다.
# gradio: 웹 기반 UI를 구축하기 위한 라이브러리
# requests: HTTP 요청을 보내기 위한 라이브러리
# io.BytesIO: 바이트 데이터를 메모리에서 파일처럼 다루기 위한 클래스
# PIL.Image, ImageDraw, ImageFont: 이미지 처리 및 그리기 위한 PIL 라이브러리
# random: 랜덤 값 생성을 위한 라이브러리
import gradio as gr
import requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
import random
import os 
import dotenv
dotenv.load_dotenv()
AZURE_FOUNDRY_KEY = os.getenv('AZURE_FOUNDRY_KEY')


# 랜덤한 색상을 생성하는 함수
# 반환값: (R, G, B) 튜플 형태의 RGB 색상 값
def random_color():
    return (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))


# Azure Computer Vision API를 사용하여 이미지 분석을 요청하는 함수
# 매개변수:
# - image_url: 분석할 이미지의 URL
# - features: 분석할 기능 목록 (예: ["objects", "caption"])
# - options: 추가 옵션 딕셔너리 (예: 성별 중립 옵션, smartCrops 비율)
# 반환값: API 응답의 JSON 데이터 또는 None (오류 시)
def request_image_analysis(image_url, features=["objects"], options=dict()):
    # Azure Computer Vision API의 엔드포인트 URL
    endpoint = "https://fimtrus-foundry999447574152744.cognitiveservices.azure.com/computervision/imageanalysis:analyze"

    # API 요청 파라미터 설정: API 버전과 분석 기능들
    params = {"api-version": "2024-02-01", "features": ",".join(features)}

    # 추가 옵션들을 파라미터에 병합
    params.update(options)

    # API 요청 헤더: 구독 키 포함
    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_FOUNDRY_KEY
    }

    # 요청 본문: 이미지 URL
    body = {"url": image_url}

    # POST 요청을 보내고 응답 받기
    response = requests.post(endpoint, params=params, headers=headers, json=body)

    # 응답이 성공적이지 않으면 None 반환
    if not response.ok:
        return None

    # 성공 시 JSON 데이터 반환
    return response.json()


# 이미지에 분석 결과를 시각적으로 그리는 함수
# 매개변수:
# - image_url: 원본 이미지 URL
# - data: API 응답 데이터 (JSON)
# 반환값: 분석 결과가 그려진 PIL Image 객체
def draw_image(image_url, data):
    # 그릴 수 있는 기능들의 키 목록
    DRAWABLE_FEATURES = ["objectsResult", "denseCaptionsResult", "smartCropsResult"]

    # 이미지 URL로부터 이미지를 다운로드하고 PIL Image로 열기
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content))

    # 이미지에 그리기 위한 Draw 객체 생성
    draw = ImageDraw.Draw(image)

    # 기본 폰트 로드 (크기 20)
    font = ImageFont.load_default(size=20)

    # 각 그릴 수 있는 기능에 대해 반복
    for feature_key in DRAWABLE_FEATURES:
        # 해당 기능의 결과 데이터 가져오기
        result = data.get(feature_key, None)
        if result is None:
            continue

        # 디버깅을 위해 결과 출력
        print(result)

        # 랜덤 색상 생성
        color = random_color()

        # 결과 값들의 목록 가져오기
        item_list = data[feature_key]["values"]

        # 각 항목에 대해 바운딩 박스 그리기
        for item in item_list:
            # 바운딩 박스 좌표 추출
            bounding_box = item["boundingBox"]
            x, y, w, h = (
                bounding_box["x"],
                bounding_box["y"],
                bounding_box["w"],
                bounding_box["h"],
            )

            # 바운딩 박스 사각형 그리기
            draw.rectangle([(x, y), (x + w, y + h)], outline=color, width=2)

            # 표시할 텍스트 초기화
            text = ""

            # 기능에 따라 텍스트 설정
            if "objectsResult" == feature_key:
                # 객체 감지 결과: 태그 이름과 신뢰도
                tag = item["tags"][0]
                name = tag["name"]
                confidence = tag["confidence"]
                text = "{}({:.2f}%)".format(name, confidence * 100)
            elif "denseCaptionsResult" == feature_key:
                # 밀집 캡션 결과: 텍스트
                text = item["text"]
            elif "smartCropsResult" == feature_key:
                # 스마트 크롭 결과: 종횡비
                text = "smartCrops : {:.2f}".format(item["aspectRatio"])

            # 텍스트를 바운딩 박스 위에 그리기
            draw.text(
                (x + 10, y),
                text,
                fill=color,
                font=font,
            )

    # 수정된 이미지 반환
    return image


# Gradio Blocks를 사용하여 웹 인터페이스 생성
with gr.Blocks() as demo:
    # 전송 버튼 클릭 시 호출되는 함수
    # 매개변수:
    # - image_url: 입력된 이미지 URL
    # - selected_features: 선택된 분석 기능들
    # - is_neutral: 성별 중립 옵션 (True/False)
    # - smart_crops_text: 스마트 크롭 비율 텍스트
    # 반환값: 그려진 이미지와 JSON 데이터
    def click_send(image_url, selected_features, is_neutral, smart_crops_text):
        # 옵션 딕셔너리 초기화
        option = dict()

        # 캡션이나 밀집 캡션이 선택된 경우 성별 중립 옵션 추가
        if "caption" in selected_features or "denseCaptions" in selected_features:
            option.update({"gender-neutral-caption": is_neutral})

        # 스마트 크롭이 선택된 경우 비율 옵션 추가
        if "smartCrops" in selected_features:
            option.update({"smartcrops-aspect-ratios": smart_crops_text})

        # API 요청 및 이미지 분석
        response_data = request_image_analysis(image_url, selected_features, option)

        # 분석 결과를 이미지에 그리기
        image = draw_image(image_url, response_data)

        # 이미지와 데이터를 반환
        return image, response_data

    # 기능 선택 변경 시 호출되는 함수
    # 매개변수: selected_features - 선택된 기능들
    # 반환값: 성별 라디오와 스마트 크롭 텍스트 박스의 업데이트
    def change_features(selected_features):
        print(selected_features)

        # 초기 가시성 설정
        selected_caption = False
        selected_smart_crops = False

        # 캡션이나 밀집 캡션이 선택되었는지 확인
        if "caption" in selected_features or "denseCaptions" in selected_features:
            selected_caption = True

        # 스마트 크롭이 선택되었는지 확인
        if "smartCrops" in selected_features:
            selected_smart_crops = True

        # 가시성 업데이트 반환
        return gr.update(visible=selected_caption), gr.update(
            visible=selected_smart_crops
        )

    # 스마트 크롭 텍스트 변경 시 호출되는 함수 (단순히 텍스트 반환)
    def change_smart_crops(text):
        return text

    # 제목 마크다운
    gr.Markdown("Image Analysis!")

    # 기능 선택을 위한 체크박스 그룹
    features_checkbox = gr.CheckboxGroup(
        label="기능 선택",
        choices=[
            "objects",  # 객체 감지
            "caption",  # 이미지 캡션 생성
            "denseCaptions",  # 밀집 캡션 (세부 설명)
            "tags",  # 태그 추출
            "smartCrops",  # 스마트 크롭 (중요 부분 추출)
            "people",  # 사람 감지
            "read",  # 텍스트 읽기 (OCR)
        ],
        value=["objects", "smartCrops"],  # 기본 선택 값
    )

    # 성별 중립 옵션을 위한 라디오 버튼 (기본적으로 숨김)
    gender_radio = gr.Radio(
        label="성별 중립 옵션",
        choices=[("중립", True), ("구분", False)],
        value=False,
        visible=False,
        interactive=True,
    )

    # 스마트 크롭 비율을 지정하는 텍스트 박스
    smart_crops_text = gr.Textbox(
        label="smartCrops 크기 지정",
        placeholder="ex) 0.75,1.2,1.7",
        visible=True,
        interactive=True,
    )

    # 이미지 URL 입력을 위한 텍스트 박스
    image_url_textbox = gr.Textbox(label="Image URL")

    # 전송 버튼
    send_button = gr.Button("전송")

    # 출력 영역: 이미지와 JSON 데이터를 나란히 표시
    with gr.Row():
        output_image = gr.Image(label="출력 이미지", interactive=False, type="pil")
        output_json = gr.JSON(label="출력 데이터")

    # 전송 버튼 클릭 이벤트 바인딩
    send_button.click(
        click_send,
        inputs=[image_url_textbox, features_checkbox, gender_radio, smart_crops_text],
        outputs=[output_image, output_json],
    )

    # 기능 체크박스 변경 이벤트 바인딩
    features_checkbox.change(
        change_features,
        inputs=[features_checkbox],
        outputs=[gender_radio, smart_crops_text],
    )

    # 스마트 크롭 텍스트 변경 이벤트 바인딩
    smart_crops_text.change(
        change_smart_crops, inputs=[smart_crops_text], outputs=[smart_crops_text]
    )

# Gradio 앱 실행
demo.launch()